In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import gzip
import os
import upsetplot

In [ ]:
INDIR= "/net/nwgc/vol1/home/czaka/analysis/mitoscope/smaht/production_analysis/integrated_variant_pipeline/v2_p25"
manifest = pd.read_csv(f'{INDIR}/merge_manifests/p25.merge.txt')
manifest['FilePath'] = manifest['FilePath'].str.split(pat=' ')

def process_item(item):
    file_name = os.path.basename(item)
    x = '-'.join(file_name.split('.')[0].split('-')[2:4])
    return x

manifest['FilePath'] = manifest['FilePath'].apply(lambda x: ','.join([process_item(item) for item in x]))
manifest['num_reps'] = manifest['FilePath'].str.count(',') + 1 

manifest.sort_values('num_reps')

In [ ]:
manifest_expanded = manifest.assign(reps=manifest['FilePath'].str.split(',')).explode('reps')
manifest_expanded[['Tech', 'Tool']] = manifest_expanded['reps'].str.split('-', expand=True)
manifest_expanded = manifest_expanded[['Donor_Tissue', 'Donor', 'Tissue_Code', 'Tech', 'Tool', 'reps']].drop_duplicates()

availability = manifest_expanded.groupby(['Donor_Tissue', 'Donor', 'Tissue_Code']).agg(
        avail_techs=("Tech", lambda x: '+'.join(sorted(x.astype(str).unique()))),
        n_avail_techs=("Tech", 'nunique'),
        avail_reps=("reps", lambda x: '+'.join(sorted(x.astype(str).unique()))),
        n_avail_reps=("reps", 'nunique')).reset_index()

availability.sort_values('n_avail_reps')

In [ ]:
tissue_availability = manifest.groupby(['Donor']).agg(
        avail_tissues=("Tissue_Code", lambda x: '+'.join(sorted(x.astype(str).unique()))),
        n_avail_tissues=("Tissue_Code", 'nunique')
)

availability = availability.merge(tissue_availability, on=['Donor'], how='left')
availability = availability.drop(columns=['Donor', 'Tissue_Code'])
availability

In [ ]:
upset_df = upsetplot.from_memberships(manifest.FilePath.str.split(","), data=manifest)
upset_df

sns.set_theme(style="ticks", context="talk", font_scale=0.8)

fig = plt.figure(figsize=(15,10))
t = upsetplot.UpSet(upset_df, element_size=None, sort_categories_by="-input", sort_by="degree", show_percentages=True, show_counts=True, totals_plot_elements=0)
t.plot(fig)
plt.suptitle("", size=14)
plt.show()

In [ ]:
blacklist=[3107]

def read_vcf(input_file):
    with gzip.open(input_file, 'rt') as fr:
        for line in fr:
            if line.startswith('#CHROM'):
                header = line.strip().lstrip('#').split('\t')
                break

    df = pd.read_csv(input_file, comment='#', sep='\t', compression='gzip', names=header)
    return df

comb_df = pd.DataFrame()

for sample in manifest['Donor_Tissue']:
    if os.path.isfile(f"{INDIR}/output/merged/{sample}.vcf.gz"):
        df = read_vcf(f"{INDIR}/output/merged/{sample}.vcf.gz")
        df['ID'] = df[['CHROM', 'POS', 'REF', 'ALT']].astype(str).agg('-'.join, axis=1)

        sample_col = [col for col in df if col.startswith('SMHT')]
        df = pd.melt(
            df[["POS", "ID"] + sample_col],
            id_vars=['POS', 'ID'],
            var_name='Sample',
            value_name='Value'
        )
        comb_df = pd.concat([comb_df, df])
    else:
        print(f'{sample} VCF is missing')
    
comb_df[['Donor', 'Tissue', 'Tech','Tool']] = comb_df['Sample'].str.split('-', expand=True)
comb_df = comb_df[comb_df['Value'] != '.:.']
comb_df[['GT', 'AF']] = comb_df['Value'].str.split(':', expand=True)
comb_df['AF'] = comb_df['AF'].astype(float)
comb_df['Donor_Tissue'] = comb_df['Donor'] + "-" + comb_df['Tissue']
comb_df['Tech_Tool'] = comb_df['Tech'] + "_" + comb_df['Tool']

comb_df = comb_df[~comb_df['POS'].isin(blacklist)]
comb_df = comb_df.drop(columns=['POS', 'Sample', 'Value', 'GT'])
comb_df


In [ ]:
tech_means = (
    comb_df
    .groupby(["Donor", "Tissue", "Donor_Tissue", "ID", "Tech"])
    .agg(mean_tech_AF=("AF", "mean"))
    .reset_index()
).pivot(index=['Donor_Tissue', 'ID'], columns='Tech', values='mean_tech_AF').reset_index()

tech_means = tech_means.fillna('.')

tech_means

In [ ]:
collapsed_df = (
    comb_df
    .groupby(["ID", "Donor", "Tissue", "Donor_Tissue"])
    .agg(
        n_tools=("Tool", "nunique"),
        tools=("Tool", lambda x: '+'.join(sorted(x.astype(str).unique()))),
        n_techs=("Tech", "nunique"),
        techs=("Tech", lambda x: '+'.join(sorted(x.astype(str).unique()))),
        n_reps=("Tech_Tool", "nunique"),
        mean_af=("AF", "mean"),
        std_af=("AF", "std"),
    ).reset_index()
)

collapsed_df = collapsed_df.merge(tech_means, on=['Donor_Tissue', 'ID'], how='left')
collapsed_df = collapsed_df.merge(availability, on='Donor_Tissue', how='left')
collapsed_df

In [ ]:
collapsed_df['tissue_reps'] = (
    collapsed_df
    .groupby(['ID', 'Donor'])['Tissue']
    .transform('count')
)

collapsed_df['tissue_rep_names'] = (
    collapsed_df
    .groupby(['ID', 'Donor'])['Tissue']
    .transform(lambda x: '+'.join(sorted(x.astype(str).unique())))
)

collapsed_df

In [ ]:
collapsed_df['CrossTech'] = np.where(collapsed_df['n_techs'] > 1, True, False)
collapsed_df['CrossTool'] = np.where(collapsed_df['n_tools'] > 1, True, False)
collapsed_df['CrossTissue'] = np.where(collapsed_df['tissue_reps'] > 1, True, False)

collapsed_df['Category_Count'] = collapsed_df[['CrossTech','CrossTool', 'CrossTissue']].sum(axis=1)

conditions = [
    (collapsed_df['Category_Count'] == 0),
    (~collapsed_df['CrossTech']) & (collapsed_df['Category_Count'] == 1),
    (collapsed_df['CrossTech']) | (collapsed_df['Category_Count'] > 1)
]

values = ['LikelyArtifact', 'LowConf', 'HighConf']
collapsed_df['tier'] = np.select(conditions, values, default='-')

collapsed_df[['CHROM', 'POS', 'REF', 'ALT']] = collapsed_df['ID'].str.split('-', expand=True)
collapsed_df['indel'] = collapsed_df.apply(lambda row: 'indel' if (len(row['REF']) > 1 or len(row['ALT']) > 1) else 'snv', axis=1)
collapsed_df['heteroplasmy_category'] = np.where(collapsed_df['mean_af'] >= 0.95, 'homo', 'hetero')
collapsed_df['variant_origin'] = np.where(collapsed_df['tissue_reps'] == collapsed_df['n_avail_tissues'], 'germline', 'somatic')
collapsed_df

In [ ]:
tier_counts = collapsed_df.groupby(['Donor','tier'])['ID'].count().unstack(fill_value=0)
tier_counts

sns.set_theme(style="ticks", context="talk", font_scale=0.8)
fig, ax = plt.subplots(figsize=(12,6))
bottom = None
order = ['HighConf', 'LowConf', 'LikelyArtifact']

#tier_counts = tier_counts.loc[tier_counts.sum(axis=1).sort_values(ascending=False).index]

for tier in order:
    ax.bar(tier_counts.index, tier_counts[tier], bottom=bottom, label=tier)
    if bottom is None:
        bottom = tier_counts[tier].copy()
    else:
        bottom += tier_counts[tier]

ax.set_ylabel('Total variants')
ax.set_xlabel('Tissue')
ax.legend(title='Tier')
plt.xticks(rotation=90)
#plt.ylim(0,90)
plt.tight_layout()


In [ ]:
# count IDs per donor/tissue/tier
counts = (
    collapsed_df
    .groupby(['Donor', 'Tissue', 'tier'])['ID']
    .count()
    .reset_index(name='count')
)

# average across tissues within each donor
tier_counts2 = (
    counts
    .groupby(['Donor', 'tier'])['count']
    .mean()
    .unstack(fill_value=0)
    .reset_index()
)

tier_counts2

sns.set_theme(style="ticks", context="talk", font_scale=0.8)

fig, ax = plt.subplots(figsize=(10, 6))

bottom = np.zeros(len(tier_counts2))
order = ['HighConf', 'LowConf', 'LikelyArtifact']

for tier in order:
    values = tier_counts2[tier]
    ax.bar(
        tier_counts2['Donor'],
        values,
        bottom=bottom,
        label=tier
    )
    bottom += values

ax.set_ylabel('Mean variant count')
ax.set_xlabel('Donor')
ax.legend(title='Tier')
plt.xticks(rotation=90, ha='center')
plt.tight_layout()

In [ ]:
tier_counts = collapsed_df.groupby(['Donor', 'Tissue', 'tier'])['ID'].count().unstack(fill_value=0).reset_index()

sns.set_theme(style="ticks", context="talk", font_scale=0.8)

order = ['HighConf', 'LowConf', 'LikelyArtifact']
donors = tier_counts['Donor'].unique()

fig, axes = plt.subplots(len(donors), 1, figsize=(10, 4 * len(donors)), sharex=False)

for ax, donor in zip(axes, donors):

    df = tier_counts[tier_counts['Donor'] == donor]
    bottom = None
    x = range(len(df))

    for tier in order:
        vals = df[tier]
        ax.bar(x, vals, bottom=bottom, label=tier)

        if bottom is None:
            bottom = vals.copy()
        else:
            bottom += vals

    ax.set_title(donor)
    ax.set_ylabel("Count")
    ax.set_xticks(x)
    ax.set_xticklabels(df["Tissue"], rotation=45)

axes[0].legend(title="Tier")
plt.tight_layout()
plt.show()

In [ ]:
cat_counts = collapsed_df.groupby(['Tissue'])[['CrossTech', 'CrossTool', 'CrossTissue']].sum().stack().reset_index()
cat_counts.columns = ['Tissue', 'Category', 'Value']
cat_counts

sns.set_theme(style="ticks", context="talk", font_scale=0.8)
fig = plt.figure(figsize=(13,6))
ax = sns.barplot(data=cat_counts, x='Tissue', y='Value', hue='Category')
sns. move_legend(ax, "lower center",bbox_to_anchor=(.5, 1), ncol=4, title=None, frameon=False) 
plt.xlabel('Tissue')
plt.ylabel('Category')
plt.show()

sns.set_theme(style="ticks", context="talk", font_scale=0.8)
sns.stripplot(cat_counts, x='Category', y='Value', hue='Category')
sns.boxplot(cat_counts, x='Category', y='Value', showfliers=False, color='white')
plt.xlabel('')
plt.ylabel('Variant count per tissue')
plt.show()



In [ ]:
sns.set_theme(style="ticks", context="talk", font_scale=0.8)

g = sns.displot(
    data=collapsed_df,
    row_order=['HighConf', 'LowConf', 'LikelyArtifact'],
    x='mean_af',
    hue='Tissue',
    row='tier',
    kind='hist',
    bins=30,
    element='step',
    stat='count',
  #  common_norm=True,
    alpha=0.6,
    height=3,
    aspect=3
)

g.set_axis_labels("Mean allele frequency", "Density")
g.set_titles(row_template="{row_name}")
plt.show()

In [ ]:
het_counts = collapsed_df.groupby(['Tissue', 'tier', 'heteroplasmy_category'])['ID'].count().unstack()
t = het_counts.stack().rename('variant_count').reset_index()

sns.set_theme(style="ticks", context="talk", font_scale=0.8)
plt.figure(figsize=(8,6))

sns.boxplot(
    data=t,
    hue='tier',
    y='variant_count',
    x='heteroplasmy_category',
    hue_order=['HighConf', 'LowConf', 'LikelyArtifact'],
    showfliers=False,
    dodge=True
)

sns.stripplot(
    data=t,
    hue='tier',
    y='variant_count',
    x='heteroplasmy_category',
    hue_order=['HighConf', 'LowConf', 'LikelyArtifact'],
    dodge=True,
    alpha=0.8,
    jitter=True,
    legend=False
)

plt.xlabel('')
plt.ylabel('Variant count per tissue')

plt.show()

In [ ]:
prob_ids = [
    'chrM-310-T-C', 'chrM-310-T-TC', 'chrM-309-C-CCT', 'chrM-302-A-AC', 'chrM-16189-T-C'
]

mask = collapsed_df['ID'].isin(prob_ids)

plt.figure(figsize=(10,8))

# background points
sns.scatterplot(
    data=collapsed_df[~mask],
    x='mean_af',
    y='std_af',
    color='blue',
    alpha=0.5,
    s=50
)

# highlighted points
sns.scatterplot(
    data=collapsed_df[mask],
    x='mean_af',
    y='std_af',
    color='red',
    alpha=0.5,
    s=50
)

plt.show()



In [ ]:
w = comb_df.groupby(['Donor', 'Tissue', 'ID', 'Tech']).agg(
    mean_af=("AF", "mean"),
    std_af=("AF", "std")
).reset_index()

plt.figure(figsize=(10,8))
sns.scatterplot(w, x='mean_af', y='std_af', hue='Tech', s=40)
w[w['std_af'] > 0.1]

#w

In [ ]:
df_wide = comb_df.pivot_table(
    index=['ID', 'Donor_Tissue'],   # keep variant + donor paired
    columns=['Tool', 'Tech'],          # or ['Tech','Tool'] if needed
    values='AF'
).reset_index()

df_wide

corr = df_wide.drop(columns=['ID','Donor_Tissue']).corr()

sns.heatmap(corr, annot=True, cmap='coolwarm', vmin=0.5, vmax=1)

plot_df = df_wide.drop(columns=['ID', 'Donor_Tissue'])

# Global limits across all variables
global_min = np.nanmin(plot_df.values)
global_max = np.nanmax(plot_df.values)

g = sns.pairplot(
    plot_df,
    corner=True,
    diag_kind=None,
    plot_kws={'alpha': 0.8, 's': 30},
)

pad = (global_max - global_min) * 0.05
# Set same limits for every subplot
for ax in g.axes.flatten():
    if ax is not None:
        ax.set_xlim(global_min - pad, global_max + pad)
        ax.set_ylim(global_min - pad, global_max + pad)

plt.show()

In [ ]:
collapsed_df.columns

In [ ]:
## write to tsv file to use w bcftools annotate
for t in collapsed_df['Donor_Tissue'].unique():
    df = collapsed_df[collapsed_df['Donor_Tissue'] == t]
    df['GT'] = 1
    df = df[['CHROM', 'POS', 'REF', 'ALT', 'tier', 'techs', 'tools', 'tissue_rep_names', 'avail_techs', 'avail_tissues', 'illumina', 'ont', 'pacbio', 'heteroplasmy_category', 'variant_origin']]
    df['POS'] = df['POS'].astype(int)
    file_path=f"{INDIR}/output/merged/annotations/{t}.tsv"
    df.sort_values('POS').to_csv(file_path, sep='\t', index=False, header=False)